# 22 Check Audio Head Trainer

In [1]:
from wooftrainer.trainers.audio_heads import (
    balanced_weights,
    choose_C,
    clip_weights,
    fit_temperature,
    make_folds,
    out_of_fold_logits,
    score_probs
)
from wooftrainer.trainers.embeddings import attach_labels, load_embeddings
from woofdecoder.branches.audio import select_layer, softmax

## Data and Model Preparation

In [2]:
x, table = load_embeddings("audio_emotion", "train", "beats_iter3_plus_as2m")
table = attach_labels(table, "audio_emotion", "train")
features = select_layer(x, "mean")

labels = table["arousal"].to_numpy()
weights = balanced_weights(labels, clip_weights(table))
folds = make_folds(labels)

## Optimizing C Value

In [3]:
best_C, c_table = choose_C(features, labels, weights, folds)
print(c_table.round(3).to_string(index=False), "\nbest C:", best_C)

 log_loss  macro_f1     C
    1.038     0.543 0.000
    0.976     0.562 0.000
    0.922     0.564 0.000
    0.890     0.572 0.001
    0.899     0.575 0.003
    0.968     0.549 0.010
    1.438     0.521 0.100
    2.415     0.515 1.000 
best C: 0.001


- at C = 1.0, the probe default settings show that the prediction is badly over fitting because log loss is super high (2.415)
- at C = 0.001, the scores show the best of both f1 (0.515 --> 0.572) and log_loss (2.415 --> 0.890).

Thus, we can trust the aurosal probabilities and the strong regularisation stopped the memorizing.

## Optimizing T Value

In [4]:
logits, classes = out_of_fold_logits(features, labels, weights, folds, best_C)
temperature = fit_temperature(logits, labels, classes, weights)
print("temperature:", round(temperature, 2))
print("before:", score_probs(labels, softmax(logits), classes, weights))
print("after:", score_probs(labels, softmax(logits/temperature), classes, weights))

temperature: 0.86
before: {'log_loss': 0.8902433376411953, 'macro_f1': 0.5715719412551769}
after: {'log_loss': 0.8871376437531784, 'macro_f1': 0.5715719412551769}
